# 07 — Ablaciones y selección (solo validación)

Implementación preparada el 2026-11-01 (America/Lima). **Etapa abierta hasta revisión de evidencia Colab de ambos regímenes.** Leer `docs/project/validacion_etapa7.md` y el cuaderno de decisiones antes de ejecutar.

Solo el usuario ejecuta modelos en Colab GPU. Sin entrenamiento, recalibración, test, Bolivia, Paraguay reservado ni demo. Las selecciones y locks se generan con mediciones reales; no existen resultados nuevos por crear este notebook.


## Hipótesis y controles previos

H1 WAIMLAP: referencia asistida > puntos DINOv3 > malla. H2: positivos mejoran hasta saturar; negativos reducen FP. H3: ajustar decoder ayuda más a eventos vistos que a Bolivia; **la parte geográfica no se contrasta aquí**.

A1: cuatro fuentes, decoder base congelado. Referencia asistida emparejada 3+/3− y distancia 16 px; se conserva además la referencia histórica 1+/1− de etapa 3. A2: k+ {1,3,5,10} con k−=3, después k− {0,1,3,5} con k+=3, recomendaciones registradas y sin reajuste de umbrales. A3: misma ruta directa de etapa 6 en ambas ramas, puntos/probe/umbrales idénticos, solo cambia decoder. No reutilizar automáticamente la ruta predictor como control de la ruta directa.

13 filas y 10 contrastes por régimen; 89/75 chips val. Todos los resultados, fallos y deltas negativos se conservan. NDWI aplazado/excluido. La selección automática usa IoU agua pooled, empate exacto por menor latencia desde imagen y luego ID estable. La calibración de etapa 5 y best de etapa 6 se registran aparte, sin nuevas búsquedas.


## 1. Entorno y paquete de fuentes

En PC ejecutar `scripts/validate_stage7_local.py` con el Python de `.venv-validation`; subir `artifacts/auto-prompt-sam2-stage7.zip` a MyDrive. Copiar su SHA en la siguiente celda. Mantener la raíz histórica completa con features y checkpoints de etapas 1–6; los ZIP de evidencia ligera no la reemplazan. Elegir runtime Colab con GPU. La instalación reconstruye el entorno fijado de Colab, no un entorno de modelos en PC.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage7')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage7.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Copiar salida de export_stage7_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Todos los modelos se recargan offline desde las generaciones autorizadas de Drive.
LOG_DIR = Path(PERSIST_ROOT) / 'stage7/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    import time
    log_path = LOG_DIR / (str(time.time_ns()) + '_' + log_name)
    print('Log persistente:', log_path)
    with log_path.open('x', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_path}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


## 2. Revisar presupuesto cerrado

Ambos regímenes comparten espacio de búsqueda y controles antes de ver resultados nuevos. No cambiar esta matriz después de inspeccionar el régimen oficial. La diferencia de componentes y total real de puntos se reporta: igualdad de k por componente no garantiza igualdad de llamadas SAM.


In [ ]:
matrix = json.loads((REPO / 'configs/ablations.yaml').read_text('utf-8'))
print(json.dumps(matrix, ensure_ascii=False, indent=2))
print('13 filas x 89/75 chips val: 1157 + 975 = 2132 unidades lógicas, con cachés previas válidas reutilizadas.')
print('Perfil: primer chip val por régimen, 13 filas, 2 rutas cache, 5 warm-up + 20 repeticiones, batch una imagen.')
print('No hay entrenamiento ni nuevos candidatos fuera de esta matriz. GPU insuficiente = registrar pendiente.')


## 3. Ejecutar prepare, resume o recover

Primera ejecución: `RUN_MODE='prepare'`. Si hay interrupción técnica, detener el escritor anterior y usar `resume` con el mismo paquete, raíz y condiciones; las generaciones completas se reutilizan y los fallos publicados no se ocultan.

Después de completar ambos regímenes, **guardar este notebook con salidas prepare**, desconectar/eliminar runtime y abrir otro runtime GPU. Repetir solo entorno y esta sección con `RUN_MODE='recover'`. Reconstruye offline desde imagen el primer chip val en cada fila exitosa, verifica todas las generaciones y recalcula selección/locks sin alterarlos. Guardar también ese notebook con salidas. No ejecutar prepare dos veces.


In [ ]:
RUN_MODE = 'prepare'  # Primera vez; luego 'resume' si se corta, o 'recover' en otro runtime.
assert RUN_MODE in ('prepare', 'resume', 'recover')
command([str(PYTHON), str(REPO / 'scripts/run_stage7.py'), RUN_MODE, '--root', PERSIST_ROOT], RUN_MODE + '.log')


## 4. Ver resultados, exportar y detener

Se informa rendimiento por chip/evento, costo del prompt (también negativo), curvas A2, diferencias A3 y fallos. Los perfiles son del primer chip val, no estiman la latencia de toda la población; incluyen lectura SHA y las sincronizaciones de etapas, con cachés OS/Drive calientes. Se separa carga inicial y ejecución con/sin features; ambas recalculan embeddings SAM. SAM y DINO permanecen residentes, incluso cuando la rama no utiliza DINO.

Compartir notebook(s) con salidas, **dos ZIP terminados y SHA**, JSON selection/lock/validation/recovery/both_regimes y logs/failure. Los locks incluyen las 13 filas finales con hashes y `stage8_authorized=false` hasta revisión externa y cierre de ambos regímenes. No abrir reservados ni seguir con la demo.


In [ ]:
for regime in matrix['regimes']:
    base = Path(PERSIST_ROOT) / 'regimes' / regime / 'stage7'
    for folder in sorted(base.iterdir()):
        evidence = folder / 'evidence'
        selection_path = evidence / 'selection.json'
        if not selection_path.is_file():
            continue
        selection = json.loads(selection_path.read_text('utf-8'))
        print('\nREGIMEN:', regime, 'estado:', selection['status'], 'seleccion:', selection['selected_id'])
        print('Costo del prompt:', selection['prompt_cost'])
        for row in selection['rows']:
            if row['status'] == 'ok':
                print(row['id'], row['metrics']['pooled']['values'],
                      'ms desde imagen:', row['efficiency']['uncached']['median_ms'])
            else:
                print(row['id'], 'FALLO:', row['failure'])
        tables = json.loads((evidence / 'validation_tables.json').read_text('utf-8'))
        print('Curvas A2:', json.dumps(tables['a2_curves'], ensure_ascii=False))
        print('H1 orden observado en val:', tables['H1_order_observed_on_validation'])
        print('H3:', tables['H3'])
        for name in ('selection', 'evaluation_lock', 'validation', 'recovery_validation', 'both_regimes'):
            file = evidence / (name + '.json')
            if file.is_file():
                print(file, 'SHA256:', hashlib.sha256(file.read_bytes()).hexdigest())
        archive = folder / 'evidence.zip'
        print('COMPARTIR:', archive, 'SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
print('Fin de etapa 7 preparada: pendiente de revision externa; no ejecutar reservados.')
